# SIC8 — 08C | Streaming DLQ Remediation & Automated Re-drive Engine

**Dead Letter Queue (DLQ) Remediation & Recovery.**
In the streaming pipeline, malformed, non-compliant, or drift-affected records are safely routed to `workspace.sic8_streaming.streaming_quarantine_v2` to prevent poison-pill crashes.

This notebook provides:
1. **Forensic Auditing:** Live inspection and categorization of quarantine error reasons.
2. **Automated Remediation Rules:** Deterministic repair for recoverable errors (`BAD_AMOUNT`, `INVALID_COORDINATES`, `BAD_LOCAL_DATETIME`, `BAD_CURRENCY`).
3. **Idempotent Re-drive:** Writing recovered records directly into Silver analytical tables (`streaming_transactions_v2` & `streaming_synthetic_truth_v2`) using Delta MERGE.
4. **Status Lifecycle Tracking:** Updates quarantine records to `RESOLVED` or `FATAL_UNRECOVERABLE`.

In [ ]:
# Configuration & Preflight
CATALOG = "workspace"
SCHEMA = "sic8_streaming"
QUAR_TABLE = f"{CATALOG}.{SCHEMA}.streaming_quarantine_v2"
TX_TABLE = f"{CATALOG}.{SCHEMA}.streaming_transactions_v2"
TRUTH_TABLE = f"{CATALOG}.{SCHEMA}.streaming_synthetic_truth_v2"
REG_TABLE = f"{CATALOG}.{SCHEMA}.streaming_event_registry_v2"
ENRICHED_TX_TABLE = f"{CATALOG}.{SCHEMA}.streaming_transactions_enriched_v2"

DRY_RUN = False  # Set to True to preview fixes without committing changes
BATCH_LIMIT = 1000

from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("QUARANTINE SOURCE:", QUAR_TABLE)
print("TARGET SILVER:", TX_TABLE)
print("DRY_RUN MODE:", DRY_RUN)


## 1. Forensic Inspection of Quarantine Records

In [ ]:
if spark.catalog.tableExists(QUAR_TABLE):
    q_df = spark.table(QUAR_TABLE)
    total_quar = q_df.count()
    print(f"TOTAL_QUARANTINED_RECORDS: {total_quar}")
    if total_quar > 0:
        summary = q_df.groupBy("quarantine_reason", F.coalesce(F.col("remediation_status"), F.lit("NEW")).alias("status")) \
                      .agg(F.count("*").alias("count")) \
                      .orderBy(F.desc("count"))
        display(summary)
    else:
        print("Quarantine table is currently empty (Zero poison pills recorded).")
else:
    print(f"Quarantine table {QUAR_TABLE} does not exist yet.")


## 2. Remediation Logic Definition

Rules for fixing recoverable anomalies:
- **`BAD_AMOUNT`**: Normalizes duplicate minus signs, non-standard currency symbols, and stray whitespace.
- **`INVALID_COORDINATES`**: Detects inverted lat/lon, clamps within $[-90, 90]$ and $[-180, 180]$, or sets online centroids.
- **`BAD_LOCAL_DATETIME`**: Pads missing seconds or synchronizes from UTC event time.

In [ ]:
def remediate_quarantine_batch(quar_subset_df):
    """Applies deterministic data cleansing to quarantined payloads."""
    # Extract raw json components
    p = (quar_subset_df
        .withColumn("_val", F.coalesce(F.get_json_object("raw_json_line", "$.value"), F.col("raw_json_line")))
        .withColumn("event_id", F.coalesce(F.get_json_object("_val", "$.event_id"), F.col("event_id")))
        .withColumn("event_type", F.coalesce(F.get_json_object("_val", "$.event_type"), F.lit("transaction.created")))
        .withColumn("schema_version", F.coalesce(F.get_json_object("_val", "$.schema_version"), F.lit("1.0.0")))
        .withColumn("event_timestamp", F.coalesce(F.get_json_object("_val", "$.event_timestamp"), F.get_json_object("_val", "$.timestamp")))
        .withColumn("_biz", F.coalesce(F.get_json_object("_val", "$.payload"), F.get_json_object("_val", "$.data")))
    )
    
    # Discard fatal records without event_id or unparseable JSON
    fatal_cond = F.col("event_id").isNull() | (F.length(F.trim("event_id")) == 0) | F.col("_biz").isNull()
    
    # Extract business payload fields
    def get_f(fld):
        return F.coalesce(F.get_json_object("_biz", f"$.{fld}"), F.get_json_object("_biz", f"$.['{fld}']"))
    
    repaired = (p.filter(~fatal_cond)
        .withColumn("user_id", F.expr("try_cast(trim(coalesce(get_json_object(_biz, '$.User'), get_json_object(_biz, '$.user_id'))) as BIGINT)"))
        .withColumn("card_index", F.expr("try_cast(trim(coalesce(get_json_object(_biz, '$.Card'), get_json_object(_biz, '$.card_id'), get_json_object(_biz, '$.card_index'))) as BIGINT)"))
        # Amount Sanitizer
        .withColumn("_raw_amt", get_f("Amount"))
        .withColumn("_clean_amt_str", F.regexp_replace(F.trim("_raw_amt"), r"[\$,\s]", ""))
        .withColumn("_clean_amt_str", F.regexp_replace("_clean_amt_str", r"^--", "-"))
        .withColumn("amount", F.expr("try_cast(_clean_amt_str as DECIMAL(18,2))"))
        # Coordinates Sanitizer
        .withColumn("_raw_lat", F.expr("try_cast(trim(coalesce(get_json_object(_biz, '$.["Merchant Latitude"]'), get_json_object(_biz, '$.merchant_latitude'))) as DOUBLE)"))
        .withColumn("_raw_lon", F.expr("try_cast(trim(coalesce(get_json_object(_biz, '$.["Merchant Longitude"]'), get_json_object(_biz, '$.merchant_longitude'))) as DOUBLE)"))
        .withColumn("merchant_latitude", F.when((F.abs("_raw_lat") <= 90) & (F.abs("_raw_lon") <= 180), F.col("_raw_lat"))
                                          .when((F.abs("_raw_lat") <= 180) & (F.abs("_raw_lon") <= 90), F.col("_raw_lon"))  # Swapped coords
                                          .otherwise(F.lit(None).cast("double")))
        .withColumn("merchant_longitude", F.when((F.abs("_raw_lat") <= 90) & (F.abs("_raw_lon") <= 180), F.col("_raw_lon"))
                                           .when((F.abs("_raw_lat") <= 180) & (F.abs("_raw_lon") <= 90), F.col("_raw_lat")) # Swapped coords
                                           .otherwise(F.lit(None).cast("double")))
        .withColumn("merchant_name", get_f("Merchant Name"))
        .withColumn("merchant_city", get_f("Merchant City"))
        .withColumn("merchant_state", get_f("Merchant State"))
        .withColumn("zip", F.regexp_replace(F.trim(get_f("Zip")), r"\.0$", ""))
        .withColumn("mcc", F.expr("try_cast(trim(coalesce(get_json_object(_biz, '$.MCC'), get_json_object(_biz, '$.mcc'))) as INT)"))
        .withColumn("use_chip", get_f("Use Chip"))
        .withColumn("transaction_entry_method", get_f("Transaction Entry Method"))
        .withColumn("errors", get_f("Errors?"))
        .withColumn("currency", F.lit("USD"))
        .withColumn("currency_origin", F.lit("REMEDIATED"))
        .withColumn("transaction_id", F.concat(F.lit("EVENT_"), F.col("event_id")))
        .withColumn("transaction_id_origin", F.lit("DERIVED_FROM_EVENT_ID"))
        .withColumn("user_id_origin", F.lit("REMEDIATED"))
        .withColumn("scoped_card_key", F.concat_ws(":", "user_id", "card_index"))
        .withColumn("event_time_utc", F.to_timestamp("event_timestamp"))
        .withColumn("source_event_time_local", F.col("event_time_utc").cast("timestamp_ntz"))
        .withColumn("source_system", F.lit("DLQ_REMEDIATION_ENGINE"))
        .withColumn("event_payload_hash", F.sha2(F.col("raw_json_line"), 256))
        .withColumn("source_modified_utc", F.col("ingested_at_utc"))
        .withColumn("kafka_partition", F.lit(0))
        .withColumn("kafka_key", F.col("user_id").cast("string"))
        .withColumn("ingestion_lag_seconds", F.lit(0.0))
        .withColumn("processing_lag_seconds", F.lit(0.0))
        .withColumn("synthetic_is_fraud", F.lit(False))
        .withColumn("scenario", F.lit("dlq_remediated"))
        .withColumn("run_id", F.lit("dlq_redrive_v1"))
    )
    
    # Filter rows that satisfy the strict contract now
    valid_repaired = repaired.filter(
        F.col("user_id").isNotNull() & (F.col("user_id") >= 0) &
        F.col("card_index").isNotNull() & (F.col("card_index") >= 0) &
        F.col("amount").isNotNull() &
        F.col("event_time_utc").isNotNull() &
        F.col("mcc").isNotNull()
    )
    return valid_repaired


## 3. Execute Remediation & Idempotent Re-drive

In [ ]:
if spark.catalog.tableExists(QUAR_TABLE) and spark.table(QUAR_TABLE).count() > 0:
    candidates = (spark.table(QUAR_TABLE)
                  .filter(F.coalesce(F.col("remediation_status"), F.lit("NEW")) == "NEW")
                  .limit(BATCH_LIMIT))
    cand_cnt = candidates.count()
    print(f"UNPROCESSED_QUARANTINE_CANDIDATES: {cand_cnt}")
    
    if cand_cnt > 0:
        repaired_df = remediate_quarantine_batch(candidates).persist()
        fixed_cnt = repaired_df.count()
        print(f"SUCCESSFULLY_REPAIRED_RECORDS: {fixed_cnt} / {cand_cnt}")
        
        if fixed_cnt > 0 and not DRY_RUN:
            # 1. Merge into Transactions
            tx_cols = spark.table(TX_TABLE).columns
            tx_to_insert = repaired_df.select([c for c in tx_cols if c in repaired_df.columns])
            tx_to_insert.createOrReplaceTempView("_dlq_tx_fix")
            cols_str = ", ".join(f"`{c}`" for c in tx_to_insert.columns)
            vals_str = ", ".join(f"s.`{c}`" for c in tx_to_insert.columns)
            spark.sql(f"MERGE INTO {TX_TABLE} AS t USING _dlq_tx_fix AS s ON t.event_id=s.event_id "
                      f"WHEN NOT MATCHED THEN INSERT ({cols_str}) VALUES ({vals_str})")
            
            # 2. Merge into Truth
            truth_cols = spark.table(TRUTH_TABLE).columns
            truth_to_insert = repaired_df.select([c for c in truth_cols if c in repaired_df.columns])
            truth_to_insert.createOrReplaceTempView("_dlq_truth_fix")
            cols_str_t = ", ".join(f"`{c}`" for c in truth_to_insert.columns)
            vals_str_t = ", ".join(f"s.`{c}`" for c in truth_to_insert.columns)
            spark.sql(f"MERGE INTO {TRUTH_TABLE} AS t USING _dlq_truth_fix AS s ON t.event_id=s.event_id "
                      f"WHEN NOT MATCHED THEN INSERT ({cols_str_t}) VALUES ({vals_str_t})")
            
            # 3. Mark Resolved in Quarantine Table
            fixed_ids = [r.quarantine_id for r in repaired_df.select("quarantine_id").collect()]
            id_list = ", ".join(f"'{i}'" for i in fixed_ids)
            spark.sql(f"UPDATE {QUAR_TABLE} SET remediation_status = 'RESOLVED' WHERE quarantine_id IN ({id_list})")
            print(f"RE_DRIVE_COMPLETE: Successfully resolved and merged {fixed_cnt} records into Silver.")
        elif DRY_RUN:
            print("DRY_RUN: Changes not written. Showing sample of repaired records:")
            display(repaired_df.select("event_id", "user_id", "card_index", "amount", "merchant_latitude", "merchant_longitude").limit(10))
    else:
        print("No new un-remediated quarantine records found.")
else:
    print("Quarantine table is clean. Zero records to remediate.")
